# Task 1: Build a Personal RAG Application

## Step 1 — Install the libraries

In [1]:
!pip install -q langchain langchain-community langchain-classic langchain-huggingface langchain-openai faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Step 2 — Import the required libraries

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_classic.text_splitter import TokenTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

/tmp/ipykernel_3077/1874898603.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## Step 3 — Load the professional information

In [3]:
loader = TextLoader("personal_info.txt")
documents = loader.load()

print(documents)

[Document(metadata={'source': 'personal_info.txt'}, page_content='PERSONAL INFORMATION\n\nName: Maymona Khanfar\nEmail: maymonakhanfar@gmail.com\nPhone: +(970) 59-295-3343\nLocation: Jenin, Palestine\nGitHub: Available on CV\nLinkedIn: Available on CV\n\n\nEDUCATION\n\nDegree: B.S. in Computer Systems Engineering\nUniversity: Palestine Technical University - Kadoorie\nDuration: Sep 2019 - Feb 2024\n\n\nGRADUATION PROJECT\n\nProject: Beauty Services Mobile Application\n\nTechnologies:\nReact Native, Expo, Redux, Node.js, Express.js, MongoDB\n\nDescription:\nDeveloped a full-stack mobile application that enables users to book appointments, search for services, browse products, and receive notifications, with a strong focus on usability and seamless user experience.\n\n\nPROFESSIONAL EXPERIENCE\n\nDATA SCIENCE & MACHINE LEARNING TRAINEE\n\nCompany: GGateaway X IBT Learning\nLocation: Remote\nDuration: Jan 2026 - Apr 2026\n\nResponsibilities:\n- Applied data analysis and visualization tech

## Step 4 — Split the document into chunks

In [4]:
splitter = TokenTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 5


## Step 5: Check the created chunks

In [5]:
# Display the created chunks

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i + 1}:")
    print(chunk.page_content)


Chunk 1:
PERSONAL INFORMATION

Name: Maymona Khanfar
Email: maymonakhanfar@gmail.com
Phone: +(970) 59-295-3343
Location: Jenin, Palestine
GitHub: Available on CV
LinkedIn: Available on CV


EDUCATION

Degree: B.S. in Computer Systems Engineering
University: Palestine Technical University - Kadoorie
Duration: Sep 2019 - Feb 2024


GRADUATION PROJECT

Project: Beauty Services Mobile Application

Technologies:
React Native, Expo, Redux, Node.js, Express.js, MongoDB

Description:
Developed a full-stack mobile application that enables users to book appointments, search for services, browse products, and receive notifications, with a strong focus on usability and seamless user experience.


PROFESSIONAL EXPERIENCE

DATA SCIENCE & MACHINE LEARNING TRAINEE

Company: GGateaway X IBT Learning
Location: Remote
Duration: Jan 2026 - Apr 2026

Responsibilities:
- Applied data analysis and visualization techniques using Excel and Power BI to extract actionable insights.
- Performed data preprocessin

## Step 6: Create embeddings

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Step 7: Create the vector database

In [7]:
vector_db = FAISS.from_documents(chunks, embeddings)

print("Vector database created successfully.")

Vector database created successfully.


## Step 8: Retrieve relevant information

In [8]:
# Search for the most 3 relevant information

query = "What technologies were used in Maymona's graduation project?"

similar_docs = vector_db.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(similar_docs):
    print(f"\nResult {i + 1}:")
    print(doc.page_content)
    print("Score:", score)


Result 1:
ibilities:
- Designed static web pages using HTML, CSS, and JavaScript.
- Focused on responsive design and clean code.


COURSES

REACT.JS & NEXT.JS COURSE

Provider: Gaza Sky Geeks
Location: Remote
Duration: Nov 2024 - Jan 2025


MARKET READY DEVELOPER PROGRAM

Provider: Gaza Sky Geeks
Location: Remote
Duration: Jul 2024 - Oct 2024


PROJECTS

PREDICTING REMOTE WORK BURNOUT

Technologies: Python, Scikit-learn
Type: Data Science Project
Organization: GGateaway X IBT Learning
Links: Available on CV


HUENEST - ONLINE HOME DECOR STORE

Technologies: React.js, Material UI
Type: Web App
Organization: Sirius Team
Link: Available on CV


KINDO - KINDERGARTEN MANAGEMENT SYSTEM

Technologies: React.js, Node.js, MongoDB
Type: Web App
Organization: Sirius Team
Link: Available on CV


CAKERYSTORE - ONLINE CAKE STORE

Technologies: React.js, Material UI
Type: Web App
Organization: Sirius Team
Link: Available on CV


ARAB CODE ACADEMY - EDUCATIONAL PLATFORM

Technologies:
Score: 1.387799

## Step 9: Create the context

In [9]:
# Create the context from the retrieved documents

context = "\n\n".join([doc.page_content for doc, score in similar_docs])

print(context)

ibilities:
- Designed static web pages using HTML, CSS, and JavaScript.
- Focused on responsive design and clean code.


COURSES

REACT.JS & NEXT.JS COURSE

Provider: Gaza Sky Geeks
Location: Remote
Duration: Nov 2024 - Jan 2025


MARKET READY DEVELOPER PROGRAM

Provider: Gaza Sky Geeks
Location: Remote
Duration: Jul 2024 - Oct 2024


PROJECTS

PREDICTING REMOTE WORK BURNOUT

Technologies: Python, Scikit-learn
Type: Data Science Project
Organization: GGateaway X IBT Learning
Links: Available on CV


HUENEST - ONLINE HOME DECOR STORE

Technologies: React.js, Material UI
Type: Web App
Organization: Sirius Team
Link: Available on CV


KINDO - KINDERGARTEN MANAGEMENT SYSTEM

Technologies: React.js, Node.js, MongoDB
Type: Web App
Organization: Sirius Team
Link: Available on CV


CAKERYSTORE - ONLINE CAKE STORE

Technologies: React.js, Material UI
Type: Web App
Organization: Sirius Team
Link: Available on CV


ARAB CODE ACADEMY - EDUCATIONAL PLATFORM

Technologies:

PERSONAL INFORMATION

Nam

## Step 10: Create the prompt

In [10]:
text = """

Answer the question using only the information provided in the context.
If the answer is not available in the context, say "The information is not available in the provided context."

Context:

{context}

Question:

{question}

Answer:

"""

template = PromptTemplate.from_template(text)

prompt = template.format(
    context=context,
    question=query
)

print(prompt)



Answer the question using only the information provided in the context.
If the answer is not available in the context, say "The information is not available in the provided context."

Context:

ibilities:
- Designed static web pages using HTML, CSS, and JavaScript.
- Focused on responsive design and clean code.


COURSES

REACT.JS & NEXT.JS COURSE

Provider: Gaza Sky Geeks
Location: Remote
Duration: Nov 2024 - Jan 2025


MARKET READY DEVELOPER PROGRAM

Provider: Gaza Sky Geeks
Location: Remote
Duration: Jul 2024 - Oct 2024


PROJECTS

PREDICTING REMOTE WORK BURNOUT

Technologies: Python, Scikit-learn
Type: Data Science Project
Organization: GGateaway X IBT Learning
Links: Available on CV


HUENEST - ONLINE HOME DECOR STORE

Technologies: React.js, Material UI
Type: Web App
Organization: Sirius Team
Link: Available on CV


KINDO - KINDERGARTEN MANAGEMENT SYSTEM

Technologies: React.js, Node.js, MongoDB
Type: Web App
Organization: Sirius Team
Link: Available on CV


CAKERYSTORE - ONLIN

## Step 11: Define the language model

In [11]:
from google.colab import userdata

api_key = userdata.get("OPENROUTER_API_KEY")

llm = ChatOpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free",
    temperature=0.2
)

## Step 12: Generate the answer

In [12]:
response = llm.invoke(prompt)

print("Answer:")
print(response.content)

print("\nSource:")
print("personal_info.txt")

Answer:
React Native, Expo, Redux, Node.js, Express.js, MongoDB

Source:
personal_info.txt
